## LoRA Strength Comparison

This experiment evaluates the effect of different **LoRA strength values** on image generation results.

All images are generated using:
- The **same prompt**
- The **same random seed**
- The **same inference settings**

Only the **LoRA adapter weight (strength)** is varied to observe how strongly the fine-tuned Megamendung style influences the output.

Higher strength values are expected to:
- Increase stylistic dominance of the LoRA
- Potentially reduce diversity or introduce artifacts


In [ ]:
import os
import torch
from diffusers import StableDiffusionPipeline
import matplotlib.pyplot as plt

assert torch.cuda.is_available(), "GPU is required to run this notebook."

OUTPUT_DIR = "../../results/lora_strength_comparison"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("✅ Environment ready")
print("📁 Output directory:", OUTPUT_DIR)


In [ ]:
# =========================
# LOAD MODEL + LoRA
# =========================
BASE_MODEL = "runwayml/stable-diffusion-v1-5"
LORA_DIR = ".../megamendung"
LORA_FILE = "(...).safetensors" # Ex. pytorch_lora_weights.safetensors

pipe_lora = StableDiffusionPipeline.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16
).to("cuda")

pipe_lora.load_lora_weights(
    LORA_DIR,
    weight_name=LORA_FILE
)

pipe_lora.enable_attention_slicing()

print("✅ LoRA loaded correctly (local path)")


In [ ]:
# =========================
# CONFIG
# =========================
PROMPT = "traditional Indonesian batik megamendung pattern, high detail, textile"
NEG_PROMPT = "blurry, low quality, distorted"
STEPS = 30
GUIDANCE = 7.5
SEED = 42
LORA_STRENGTHS = [0.6, 1.0, 1.4]

# =========================
# GENERATE IMAGES
# =========================
images = []

for strength in LORA_STRENGTHS:
    pipe_lora.set_adapters(["default_0"], adapter_weights=[strength])

    generator = torch.Generator("cuda").manual_seed(SEED)

    image = pipe_lora(
        PROMPT,
        negative_prompt=NEG_PROMPT,
        num_inference_steps=STEPS,
        guidance_scale=GUIDANCE,
        generator=generator
    ).images[0]
    image.save(f"{OUTPUT_DIR}/lora_strength_{strength}.png")

    images.append(image)

# =========================
# PLOT RESULTS
# =========================
plt.figure(figsize=(15, 5))

for i, (img, strength) in enumerate(zip(images, LORA_STRENGTHS)):
    plt.subplot(1, 3, i + 1)
    plt.imshow(img)
    plt.title(f"LoRA strength = {strength}")
    plt.axis("off")

plt.suptitle("LoRA Strength Comparison (Same Prompt & Seed)", fontsize=16)
plt.show()
